# 07 方法论扩展演示

**Simulated data methodology demonstration — not used for the primary experiment conclusion.** 全部为模拟用户级数据；不读取、不回传主实验任何数值或参数。

> **为什么需要本 Notebook**：Udacity 原数据是日粒度聚合，缺少用户级实验前协变量、渠道/设备字段，主项目不做真实 CUPED/HTE/用户级 AA。这里用**完全模拟**的用户级数据，演示这些方法的机制与方差削减幅度。
>
> **独立目录**：本 Notebook 使用自含随机种子与自定 DGP 参数，**不读取**主项目 config 与 data/processed 下任何结果，**只写入** `methodology_demo/` 目录；任何数字都不进入主实验结论（不参与主结论）。

## 模拟用户级实验数据 DGP
**Simulated data methodology demonstration — not used for the primary experiment conclusion.** 全部为模拟用户级数据；不读取、不回传主实验任何数值或参数。
- 用户 i=1..n；实验前协变量 X~N(0,1)（如实验前 30 天活跃时长，标准化）；处理 W~Bernoulli(0.5) 随机分配。
- 结果模型 **Y = τ·W + β·X + ε，ε~N(0,σ²)**；X–Y 相关性由 β 控制：ρ(X,Y)=β/√(β²+σ²)（τ 很小时近似）。
- 参数（仅本演示用，与主项目无关）：n=10,000、σ=1、β=1（ρ≈0.707）、演示效应 τ=0.05。

In [1]:
# DGP（自含，不读主项目配置）
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20260901            # 本演示自有种子
rng = np.random.default_rng(SEED)
N_USERS = 10_000; SIGMA = 1.0; BETA = 1.0; TAU_DEMO = 0.05
OUT = Path("methodology_demo"); OUT.mkdir(exist_ok=True)

def dgp(n=N_USERS, tau=TAU_DEMO, beta=BETA, sigma=SIGMA, rng=rng):
    X = rng.normal(0, 1, n)
    W = rng.integers(0, 2, n)
    eps = rng.normal(0, sigma, n)
    Y = tau*W + beta*X + eps
    return pd.DataFrame({"X": X, "W": W, "Y": Y})

df = dgp()
rho = BETA/np.sqrt(BETA**2+SIGMA**2)
print("单次模拟样本：", df.shape, "| 理论 corr(X,Y) ≈ %.4f" % rho,
      "| 实测 corr = %.4f" % df[["X","Y"]].corr().iloc[0,1])

单次模拟样本： (10000, 3) | 理论 corr(X,Y) ≈ 0.7071 | 实测 corr = 0.7076


## CUPED 协变量调整演示（naive vs adjusted）
**Simulated data methodology demonstration — not used for the primary experiment conclusion.** 全部为模拟用户级数据；不读取、不回传主实验任何数值或参数。
- θ = Cov(Y,X)/Var(X)（X 为实验前变量、与 W 独立，可用全样本估计）；调整后 **Y_adj = Y − θ(X−X̄)**。
- 对比 naive 差均值估计与 CUPED 估计的 θ、SE、CI 宽度；理论方差削减 ≈ ρ²。重复 S=2,000 次模拟给出经验方差削减与 95% CI 覆盖率。

In [2]:
# 单次数据上的估计对比
def estimate(dat):
    c, e = dat[dat.W == 0], dat[dat.W == 1]
    # naive
    tau_hat = e.Y.mean()-c.Y.mean()
    se_naive = np.sqrt(e.Y.var(ddof=1)/len(e) + c.Y.var(ddof=1)/len(c))
    # CUPED（theta 用全样本，X 实验前、与 W 独立）
    theta = np.cov(dat.Y, dat.X, ddof=1)[0,1]/np.var(dat.X, ddof=1)
    yadj = dat.Y - theta*(dat.X-dat.X.mean())
    ca, ea = yadj[dat.W == 0], yadj[dat.W == 1]
    tau_c = ea.mean()-ca.mean()
    se_c = np.sqrt(ea.var(ddof=1)/len(ea) + ca.var(ddof=1)/len(ca))
    return tau_hat, se_naive, tau_c, se_c, theta

th, sn, tc, sc, theta = estimate(df)
print("theta = %.4f（理论=β=1）" % theta)
print("naive  : tau=%.4f SE=%.4f CI 宽度=%.4f" % (th, sn, 2*1.96*sn))
print("CUPED  : tau=%.4f SE=%.4f CI 宽度=%.4f" % (tc, sc, 2*1.96*sc))
print("单次方差削减 = %.2f%%（理论 rho^2=%.2f%%）" % ((1-sc**2/sn**2)*100, rho**2*100))

theta = 1.0015（理论=β=1）
naive  : tau=0.0447 SE=0.0281 CI 宽度=0.1101
CUPED  : tau=0.0565 SE=0.0198 CI 宽度=0.0777
单次方差削减 = 50.10%（理论 rho^2=50.00%）


In [3]:
# S=2000 次重复：经验 SE、方差削减、95%CI 覆盖率
S = 2000
rows = []
g = np.random.default_rng(SEED+7)
for _ in range(S):
    d = dgp(rng=g)
    th, sn, tc, sc, _ = estimate(d)
    rows.append((th, sn, tc, sc))
sim = np.array(rows)
naive_est, naive_se, cuped_est, cuped_se = sim.T
vr = 1-cuped_est.var(ddof=1)/naive_est.var(ddof=1)
cov_n = np.mean((naive_est-1.96*naive_se < TAU_DEMO) & (naive_est+1.96*naive_se > TAU_DEMO))
cov_c = np.mean((cuped_est-1.96*cuped_se < TAU_DEMO) & (cuped_est+1.96*cuped_se > TAU_DEMO))
cuped_summary = {"theta_theory_beta": BETA, "rho2_theory_variance_reduction": rho**2,
                 "empirical_variance_reduction": float(vr),
                 "naive_empirical_SE": float(naive_est.std(ddof=1)),
                 "cuped_empirical_SE": float(cuped_est.std(ddof=1)),
                 "naive_CI95_coverage": float(cov_n), "cuped_CI95_coverage": float(cov_c),
                 "mean_CI_width_ratio": float((2*1.96*cuped_se).mean()/(2*1.96*naive_se).mean())}
print("经验方差削减 = %.2f%%（理论 %.2f%%）" % (vr*100, rho**2*100))
print("经验 SE: naive=%.4f CUPED=%.4f；平均 CI 宽度比 = %.3f" %
      (naive_est.std(ddof=1), cuped_est.std(ddof=1), cuped_summary["mean_CI_width_ratio"]))
print("95%%CI 覆盖率: naive=%.3f CUPED=%.3f（均应≈0.95）" % (cov_n, cov_c))

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.hist(naive_est, bins=60, alpha=.6, density=True, label="naive estimator")
ax.hist(cuped_est, bins=60, alpha=.6, density=True, label="CUPED-adjusted")
ax.axvline(TAU_DEMO, color="black", ls="--", label=f"true tau={TAU_DEMO}")
ax.set_title("CUPED variance reduction (simulated; methodology demo)")
ax.set_xlabel("estimated treatment effect"); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT/"fig_cuped_variance.png"); plt.close(fig)
print("saved methodology_demo/fig_cuped_variance.png")

经验方差削减 = 50.61%（理论 50.00%）
经验 SE: naive=0.0287 CUPED=0.0202；平均 CI 宽度比 = 0.707
95%CI 覆盖率: naive=0.946 CUPED=0.947（均应≈0.95）


saved methodology_demo/fig_cuped_variance.png


In [4]:
# 落盘到隔离目录（不写 data/processed）
(OUT/"sim_summary.json").write_text(json.dumps(
    {"disclaimer": "Simulated data methodology demonstration - not used for the primary experiment conclusion",
     "dgp": {"n": N_USERS, "sigma": SIGMA, "beta": BETA, "tau_demo": TAU_DEMO, "rho_theory": rho},
     "cuped": cuped_summary}, indent=2), encoding="utf-8")
back = json.loads((OUT/"sim_summary.json").read_text(encoding="utf-8"))
assert back["cuped"]["empirical_variance_reduction"] > 0.4
print("methodology_demo/sim_summary.json (43-44) written & re-read OK")

methodology_demo/sim_summary.json (43-44) written & re-read OK


## （并入）｜ 模拟零效应假阳性测试
**Simulated data methodology demonstration — not used for the primary experiment conclusion.** 全部为模拟用户级数据；不读取、不回传主实验任何数值或参数。
设定真实处理效应 **τ=0**，重复 S=2,000 次跑同一套 naive / CUPED 检验流程，观察 α=0.05 下经验拒绝率是否≈0.05，并看 p 值是否均匀。**这是统计流程的方法演示，不是对 Udacity 原实验随机化质量的实证验证（更不是 AA Test）。**

In [5]:
# tau=0 下的经验假阳性率（naive 与 CUPED 各 2000 次）
from scipy import stats as st
S0 = 2000
p_naive, p_cuped = [], []
g0 = np.random.default_rng(SEED+23)
for _ in range(S0):
    d = dgp(tau=0.0, rng=g0)
    th, sn, tc, sc, _ = estimate(d)
    p_naive.append(2*st.norm.sf(abs(th/sn)))
    p_cuped.append(2*st.norm.sf(abs(tc/sc)))
p_naive, p_cuped = np.array(p_naive), np.array(p_cuped)
fpr_n = float(np.mean(p_naive < 0.05)); fpr_c = float(np.mean(p_cuped < 0.05))
mc = 1.96*np.sqrt(.05*.95/S0)
print("零效应经验拒绝率@α=0.05：naive=%.4f，CUPED=%.4f（设计值 0.05，MC 包络 %.3f~%.3f）"
      % (fpr_n, fpr_c, .05-mc, .05+mc))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), dpi=150, sharey=True)
for ax, p, name in zip(axes, [p_naive, p_cuped], ["naive", "CUPED"]):
    ax.hist(p, bins=20, color="#4c78a8", alpha=.8)
    ax.axhline(S0/20, color="black", ls="--", lw=1, label="uniform expectation")
    ax.set_title(f"{name}: p-values under tau=0 (FPR={np.mean(p<.05):.3f})")
    ax.set_xlabel("p-value"); ax.legend(fontsize=8); ax.grid(alpha=.3)
axes[0].set_ylabel("count")
fig.suptitle("Zero-effect false-positive demo (simulated; not randomization-quality validation)")
fig.tight_layout(); fig.savefig(OUT/"fig_zero_effect_fpr.png"); plt.close(fig)
print("saved methodology_demo/fig_zero_effect_fpr.png")

零效应经验拒绝率@α=0.05：naive=0.0525，CUPED=0.0560（设计值 0.05，MC 包络 0.040~0.060）
saved methodology_demo/fig_zero_effect_fpr.png


In [6]:
# 追加落盘
summ = json.loads((OUT/"sim_summary.json").read_text(encoding="utf-8"))
summ["zero_effect"] = {"true_tau": 0.0, "repeats": S0, "alpha": 0.05,
                              "naive_empirical_FPR": fpr_n, "cuped_empirical_FPR": fpr_c,
                              "note": "method demo only; NOT validation of original randomization; NOT an AA test"}
(OUT/"sim_summary.json").write_text(json.dumps(summ, indent=2), encoding="utf-8")
assert json.loads((OUT/"sim_summary.json").read_text())["zero_effect"]["naive_empirical_FPR"] == fpr_n
print("sim_summary.json updated with zero_effect")

sim_summary.json updated with zero_effect


## HTE 演示：子组效应与 treatment × feature 交互
**Simulated data methodology demonstration — not used for the primary experiment conclusion.** 全部为模拟用户级数据；不读取、不回传主实验任何数值或参数。
- 模拟二分类用户特征 G~Bernoulli(0.5)（如设备/渠道占位）；模型 **Y=τ·W+β·X+γ·(W·G)+ε**。
- 参数（仅演示）：τ=0.02（G=0 子组效应）、γ=0.08（交互项），故 G=1 子组效应=τ+γ=0.10。
- 展示：单次数据的子组效应估计 + 交互项回归；S=1,000 次重复验证交互估计无偏与覆盖率。不做 CATE/causal tree。

In [7]:
# HTE DGP + 单次估计
import statsmodels.formula.api as smf
TAU0, GAMMA = 0.02, 0.08
def dgp_hte(n=N_USERS, tau=TAU0, gamma=GAMMA, beta=BETA, sigma=SIGMA, rng=rng):
    X = rng.normal(0,1,n); W = rng.integers(0,2,n); G = rng.integers(0,2,n)
    Y = tau*W + beta*X + gamma*W*G + rng.normal(0,sigma,n)
    return pd.DataFrame({"X":X,"W":W,"G":G,"Y":Y})

dh = dgp_hte()
sub = dh.groupby(["G","W"]).Y.agg(["mean","count"])
eff_g0 = dh[(dh.G==0)&(dh.W==1)].Y.mean()-dh[(dh.G==0)&(dh.W==0)].Y.mean()
eff_g1 = dh[(dh.G==1)&(dh.W==1)].Y.mean()-dh[(dh.G==1)&(dh.W==0)].Y.mean()
m = smf.ols("Y ~ W + G + W:G + X", data=dh).fit()
print("单次：G=0 子组效应=%.4f（真0.02）；G=1 子组效应=%.4f（真0.10）" % (eff_g0, eff_g1))
print("交互项 W:G 系数=%.4f（真 γ=0.08），p=%.4f" % (m.params["W:G"], m.pvalues["W:G"]))

单次：G=0 子组效应=0.0445（真0.02）；G=1 子组效应=0.1270（真0.10）
交互项 W:G 系数=0.1224（真 γ=0.08），p=0.0023


In [8]:
# S=1000 次重复：交互估计无偏性 + 95%CI 覆盖率
S_h = 1000
gh = np.random.default_rng(SEED+45)
g_hat, g_se, cover, sg0, sg1 = [], [], [], [], []
for _ in range(S_h):
    d = dgp_hte(rng=gh)
    mm = smf.ols("Y ~ W + G + W:G + X", data=d).fit()
    g_hat.append(mm.params["W:G"]); g_se.append(mm.bse["W:G"])
    ci = mm.conf_int().loc["W:G"]; cover.append(ci[0] < GAMMA < ci[1])
    sg0.append(d[(d.G==0)&(d.W==1)].Y.mean()-d[(d.G==0)&(d.W==0)].Y.mean())
    sg1.append(d[(d.G==1)&(d.W==1)].Y.mean()-d[(d.G==1)&(d.W==0)].Y.mean())
g_hat, g_se = np.array(g_hat), np.array(g_se)
hte_summary = {"true_gamma": GAMMA, "mean_gamma_hat": float(g_hat.mean()),
               "empirical_SE": float(g_hat.std(ddof=1)), "coverage95": float(np.mean(cover)),
               "mean_subgroup_effect_G0": float(np.mean(sg0)),
               "mean_subgroup_effect_G1": float(np.mean(sg1))}
print("交互估计均值=%.4f（真0.08），经验SE=%.4f，95%%CI覆盖率=%.3f" %
      (g_hat.mean(), g_hat.std(ddof=1), np.mean(cover)))
print("重复抽样子组效应均值：G0=%.4f，G1=%.4f" % (np.mean(sg0), np.mean(sg1)))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), dpi=150)
axes[0].hist(sg0, bins=40, alpha=.7, label="G=0 effect (true .02)")
axes[0].hist(sg1, bins=40, alpha=.7, label="G=1 effect (true .10)")
axes[0].legend(fontsize=8); axes[0].set_title("Subgroup effects across simulations"); axes[0].grid(alpha=.3)
axes[1].hist(g_hat, bins=40, color="#59a14f", alpha=.8)
axes[1].axvline(GAMMA, color="black", ls="--", label="true gamma=.08")
axes[1].legend(fontsize=8); axes[1].set_title("W x G interaction estimator"); axes[1].grid(alpha=.3)
fig.suptitle("HTE demo (simulated; methodology demonstration)")
fig.tight_layout(); fig.savefig(OUT/"fig_hte_subgroups.png"); plt.close(fig)
print("saved methodology_demo/fig_hte_subgroups.png")

交互估计均值=0.0795（真0.08），经验SE=0.0410，95%CI覆盖率=0.941
重复抽样子组效应均值：G0=0.0199，G1=0.0985


saved methodology_demo/fig_hte_subgroups.png


In [9]:
# 追加落盘
summ = json.loads((OUT/"sim_summary.json").read_text(encoding="utf-8"))
summ["hte"] = hte_summary
(OUT/"sim_summary.json").write_text(json.dumps(summ, indent=2), encoding="utf-8")
print("sim_summary.json updated with hte")

sim_summary.json updated with hte


## 多重检验演示：未校正 vs Bonferroni vs Benjamini–Hochberg(FDR)
**Simulated data methodology demonstration — not used for the primary experiment conclusion.** 全部为模拟用户级数据；不读取、不回传主实验任何数值或参数。
- 场景 A（全局零假设）：每次实验模拟 K=40 个独立子组检验（z~N(0,1)），重复 S=2,000 次，比较三种口径的 FWER/FDR。
- 场景 B（部分真效应）：40 个里 8 个有真实效应（z 均值 2.8），比较平均发现数（检验功效视角）。

In [10]:
# 多重检验模拟（纯 z/p 值，快速可复现）
from statsmodels.stats.multitest import multipletests
K = 40; S_m = 2000; K_TRUE = 8; TRUE_Z = 2.8
gm = np.random.default_rng(SEED+46)

def one_experiment(global_null):
    z = gm.normal(0, 1, K)
    if not global_null:
        z[:K_TRUE] += TRUE_Z
    p = 2*st.norm.sf(np.abs(z))
    rej_raw = p < 0.05
    rej_bonf = multipletests(p, method="bonferroni")[0]
    rej_bh = multipletests(p, method="fdr_bh")[0]
    return p, rej_raw, rej_bonf, rej_bh

def simulate(global_null, S=S_m):
    fwer_raw=[]; fwer_bonf=[]; fdr_bh=[]; disc = {"raw":[],"bonf":[],"bh":[]}
    for _ in range(S):
        _, r0, rb, rh = one_experiment(global_null)
        if global_null:
            fwer_raw.append(r0.any()); fwer_bonf.append(rb.any())
            fdr_bh.append(rh.sum()/max(rh.sum(),1) if rh.any() else 0.0)
        disc["raw"].append(r0.sum()); disc["bonf"].append(rb.sum()); disc["bh"].append(rh.sum())
    out = {"avg_discoveries": {k: float(np.mean(v)) for k, v in disc.items()}}
    if global_null:
        out.update({"FWER_unadjusted": float(np.mean(fwer_raw)),
                    "FWER_bonferroni": float(np.mean(fwer_bonf)),
                    "FDR_BH": float(np.mean(fdr_bh)),
                    "theory_FWER_unadjusted": 1-0.95**K})
    return out

null_res = simulate(True); mixed_res = simulate(False)
print("场景A 全局零假设（K=40，名义α=0.05）：")
print("  未校正 FWER=%.3f（理论 1-.95^40=%.3f）| Bonferroni FWER=%.3f | BH FDR=%.3f"
      % (null_res["FWER_unadjusted"], null_res["theory_FWER_unadjusted"],
         null_res["FWER_bonferroni"], null_res["FDR_BH"]))
print("场景B 8/40 真效应：平均发现数 raw=%.2f / Bonferroni=%.2f / BH=%.2f（真效应数=8）"
      % (mixed_res["avg_discoveries"]["raw"], mixed_res["avg_discoveries"]["bonf"],
         mixed_res["avg_discoveries"]["bh"]))

场景A 全局零假设（K=40，名义α=0.05）：
  未校正 FWER=0.860（理论 1-.95^40=0.871）| Bonferroni FWER=0.050 | BH FDR=0.050
场景B 8/40 真效应：平均发现数 raw=7.97 / Bonferroni=2.72 / BH=4.38（真效应数=8）


In [11]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), dpi=150)
labels = ["unadjusted", "Bonferroni", "BH-FDR"]
axes[0].bar(labels, [null_res["FWER_unadjusted"], null_res["FWER_bonferroni"], null_res["FDR_BH"]],
            color=["#d62728","#4c78a8","#2ca02c"], alpha=.8)
axes[0].axhline(.05, color="black", ls="--", lw=1, label="nominal 0.05")
axes[0].set_title("Global null: FWER / FDR (K=40)"); axes[0].legend(fontsize=8); axes[0].grid(axis="y",alpha=.3)
axes[0].set_ylim(0,1)
axes[1].bar(labels, [mixed_res["avg_discoveries"]["raw"], mixed_res["avg_discoveries"]["bonf"],
                     mixed_res["avg_discoveries"]["bh"]], color=["#d62728","#4c78a8","#2ca02c"], alpha=.8)
axes[1].axhline(K_TRUE, color="black", ls="--", lw=1, label="true effects=8")
axes[1].set_title("8/40 true effects: avg discoveries"); axes[1].legend(fontsize=8); axes[1].grid(axis="y",alpha=.3)
fig.suptitle("Multiple-testing demo (simulated; methodology demonstration)")
fig.tight_layout(); fig.savefig(OUT/"fig_multiplicity.png"); plt.close(fig)
print("saved methodology_demo/fig_multiplicity.png")

saved methodology_demo/fig_multiplicity.png


In [12]:
# 追加落盘
summ = json.loads((OUT/"sim_summary.json").read_text(encoding="utf-8"))
summ["multiplicity"] = {"K": K, "repeats": S_m, "global_null": null_res, "mixed": mixed_res}
(OUT/"sim_summary.json").write_text(json.dumps(summ, indent=2), encoding="utf-8")
print("sim_summary.json updated with multiplicity")

sim_summary.json updated with multiplicity


## 免责声明与独立目录核验
**Simulated data methodology demonstration — not used for the primary experiment conclusion.** 全部为模拟用户级数据；不读取、不回传主实验任何数值或参数。
- **说明**：本 Notebook 开头与每一节结果均已标注上述免责声明；所有 DGP 参数（n、τ、β、γ、K、真效应个数等）均为本演示自定，与 Udacity 真实数据无任何数值往来。
- **（独立目录）**：本 Notebook 不 import/读取主项目 config 与 data/processed 下任何文件；所有写入仅发生在 `methodology_demo/`。下面用代码自查输出位置；主结果 JSON 哈希不变由仓库侧在提交前比对。

In [13]:
# 隔离自查：本 Notebook 运行后新增/修改的文件只能在 methodology_demo/
written = sorted(p.name for p in OUT.iterdir() if p.is_file() and p.name != ".gitkeep")
print("methodology_demo/ 内文件：", written)
assert all(n.startswith("fig") or n == "sim_summary.json" for n in written)
# 全局命名空间中不得出现主项目结果对象
for forbidden in ["main", "ni_decision", "cost_benefit", "design_precision", "quality_checks", "CFG"]:
    assert forbidden not in dir(), f"leakage: {forbidden}"
print("隔离自查通过：未引用主分析对象；产物仅在 methodology_demo/。")
print("\n最终免责声明：Simulated data methodology demonstration — not used for the primary experiment conclusion.")

methodology_demo/ 内文件： ['fig_cuped_variance.png', 'fig_hte_subgroups.png', 'fig_multiplicity.png', 'fig_zero_effect_fpr.png', 'sim_summary.json']
隔离自查通过：未引用主分析对象；产物仅在 methodology_demo/。

最终免责声明：Simulated data methodology demonstration — not used for the primary experiment conclusion.


### 小结（全部为模拟演示，不参与主结论）
1. **CUPED**：ρ≈0.707 时经验方差削减约 50.6%（理论 ρ²=50%），CI 宽度缩到 0.707 倍，覆盖率保持 0.95——若未来能采集用户级实验前协变量，可按此机制提效。
2. **零效应假阳性**：naive/CUPED 在 τ=0 下经验拒绝率 0.0525/0.0560，符合 α=0.05 设计；这是流程演示，不是原实验随机化质量验证、不是 AA Test。
3. **HTE**：treatment×feature 交互估计无偏（0.0795 vs 真 0.08）、覆盖率 0.941，演示具备用户特征后如何做子组/交互分析。
4. **多重检验**：K=40 时不校正 FWER 高达 0.86，Bonferroni 与 BH 均控住错误率（0.05），BH 在 8 个真效应场景下发现数（4.38）多于 Bonferroni（2.72），体现 FDR 控制的功效优势。